In [ ]:
import os
import math
from collections import defaultdict

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.windows import Window
from tqdm.auto import tqdm


In [ ]:
def window_grid(width: int, height: int, blockx: int, blocky: int):
    """
    Generate windows to process raster in chunks, avoiding memory overflow on large datasets.
    """
    for row_off in range(0, height, blocky):
        # Handle edge case where final block may be smaller than blocky
        h = min(blocky, height - row_off)
        for col_off in range(0, width, blockx):
            # Handle edge case where final block may be smaller than blockx
            w = min(blockx, width - col_off)
            yield Window(col_off=col_off, row_off=row_off, width=w, height=h)
            

def total_windows(width: int, height: int, blockx: int, blocky: int) -> int:
    """
    Calculate total number of windows needed for progress tracking.
    """
    return math.ceil(width / blockx) * math.ceil(height / blocky)


def compute_extents_both(
    access_unmasked_path: str,
    mask_path: str,
    state_raster_path: str,
    state_lookup_table: dict[int, str],
    blockx: int = 1024,
    blocky: int = 1024,
    pixel_area_m2: float = 900.0,
    exclude_state0: bool = True,
    ) -> pd.DataFrame:
    """
    Compute spatial extents of access classes (roadless/wilderness/roaded) per state.
    
    Args:
        access_unmasked_path: Path to 3-band raster with access proportions (0-100%)
        mask_path: Path to binary mask defining study area (e.g., forest cover)
        state_raster_path: Path to state boundary raster (integer state IDs)
        state_lookup_table: Mapping from state ID to human-readable name
        blockx, blocky: Window size for block processing (tune based on available memory)
        pixel_area_m2: Area represented by each pixel (default 900 = 30m resolution)
        exclude_state0: Whether to exclude state ID 0 (typically ocean/background)
    
    Returns:
        DataFrame with columns: state_name, access_class, extent_km2, extent_masked_km2
    """

    with rasterio.open(access_unmasked_path) as au_ds, /
         rasterio.open(mask_path) as mask_ds, /
         rasterio.open(state_raster_path) as s_ds:

        if au_ds.count != 3:
            raise ValueError("Access raster must have 3 bands.")

        # Attempt to use embedded metadata for band names; fallback to convention if unavailable
        band_names = (list(au_ds.descriptions)
                      if au_ds.descriptions and any(au_ds.descriptions)
                      else ["roadless", "wilderness", "roaded"])
        if len(band_names) != 3:
            band_names = ["roadless", "wilderness", "roaded"]

        # Track per-state statistics separately for unmasked (all valid pixels) and masked (forest-only)
        per_state_u_m2 = defaultdict(lambda: np.zeros(3, dtype=np.float64))
        per_state_m_m2 = defaultdict(lambda: np.zeros(3, dtype=np.float64))

        tot = total_windows(au_ds.width, au_ds.height, blockx, blocky)
        for win in tqdm(window_grid(au_ds.width, au_ds.height, blockx, blocky),
                        total=tot, desc="Blocks", leave=False, mininterval=1.0):

            acc_u = au_ds.read(indexes=[1, 2, 3], window=win, masked=False)
            
            # Skip entirely empty blocks to save computation time
            if not acc_u.any():
                continue

            states = s_ds.read(1, window=win, masked=False)
            mask = mask_ds.read(1, window=win, masked=False)

            # Build validity masks: state 0 typically represents ocean/background/invalid regions
            valid_state = (states != 0) if exclude_state0 else np.ones_like(states, dtype=bool)
            forest_valid = (mask > 0)  

            # Unmasked analysis: exclude only invalid geographic regions (state 0)
            vmask_u = valid_state
            if not vmask_u.any():
                continue

            # Masked analysis: further restrict to study-relevant pixels (e.g., forest-only areas)
            vmask_m = valid_state & forest_valid

            # Convert from percent (0-100) to proportion (0-1) for area calculations
            props_u = acc_u.astype(np.float64) / 100.0

            # Accumulate statistics per state, handling both analysis types
            for sid in np.unique(states[vmask_u]):
                # Unmasked: all valid geographic pixels for this state
                m_u = (states == sid) & vmask_u
                if m_u.any():
                    per_state_u_m2[int(sid)] += props_u[:, m_u].sum(axis=1) * pixel_area_m2

                # Masked: only study-relevant pixels for this state
                m_m = (states == sid) & vmask_m
                if m_m.any():
                    per_state_m_m2[int(sid)] += props_u[:, m_m].sum(axis=1) * pixel_area_m2

        # Convert accumulated statistics into tidy DataFrame format (one row per state-access combination)
        rows = []

        def push_rows(state_name: str, arr_u_m2: np.ndarray, arr_m_m2: np.ndarray):
            """Convert state's 3-band array into individual rows with km² units."""
            arr_u_km2 = arr_u_m2 / 1_000_000.0
            arr_m_km2 = arr_m_m2 / 1_000_000.0
            for i, bname in enumerate(band_names):
                rows.append({
                    "state_name": state_name,
                    "access_class": bname,
                    "extent_km2": float(arr_u_km2[i]),
                    "extent_masked_km2": float(arr_m_km2[i]),
                })

        # Process each state (excluding background/ocean represented by state 0)
        for sid in sorted(k for k in per_state_u_m2.keys() if k != 0):
            sname = state_lookup_table.get(int(sid), f"STATE_{sid}")
            push_rows(sname, per_state_u_m2[sid], per_state_m_m2[sid])

        df = pd.DataFrame(rows)

        # Apply semantic ordering to access classes for consistent analysis/visualization
        desired_order = ["roaded", "roadless", "wilderness"]
             
        # Gracefully handle missing or extra access classes in the data
        order = [c for c in desired_order if c in set(df["access_class"])]
        others = [c for c in df["access_class"].unique() if c not in order]
        cat = pd.Categorical(df["access_class"], categories=order + others, ordered=True)
        df["access_class"] = cat

        return df.sort_values(["state_name", "access_class"]).reset_index(drop=True)
    

### File I/O

In [ ]:
# Define the path to the project repo
project_folder = "<PATH/TO/PROJECT/FOLDER>"

# NFS access raster path
access_unmasked_path = f"{project_folder}/data/rasters/nfs_access_proportions.tif"

# NLCD-derived forest/no-forest mask
mask_path = f"{project_folder}/data/rasters/ncld_forest_mask/nlcd_forest_mask_1984_2024_mosaic.tif"

# Rasterized state boundaries
state_raster_path = f"{project_folder}/data/rasters/western_states.tif"

# Load in reference state shapefile (allows us to map the interger names back to the State's actual name)
state_shp_path = f"{project_folder}/data/shapefiles/westernstates.shp"

# Define the output CSV Path
out_csv = f"{project_folder}/data/tables/nfs_access_extent_km2.csv"


### Compute the area statistics

In [ ]:
# Define the statistics
blockx = 1024
blocky = 1024
pixel_area_m2 = 900.0

In [ ]:
# Foramt the reference state shapefile
states = gpd.read_file(state_shp_path)[["STATE_INT", "NAME"]]
state_lookup_table = dict(zip(states["STATE_INT"].astype(int), states["NAME"])))

In [ ]:
# Compute the extents
output_df = compute_extents_both(
    access_unmasked_path,
    mask_path, 
    state_raster_path,
    state_lookup_table,
    blockx = blockx,
    blocky = blocky,
    pixel_area_m2 = pixel_area_m2,
    exclude_state0 = True,
)

output_df.to_csv(out_csv, index=False)
print(f"Wrote: {out_csv}"